<div style="background: linear-gradient(120deg, #1ca9c9 0%, #40c4d6 60%, #7fe0e0 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.2);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Actividad 3: Extracción de archivos Parquet y JSON</h1>
        <h3 style="margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;">Laboratorio de Procesamiento de Datos</h3>
    </div>
</div>


# 1. De múltiples CSV a un dataset Parquet analítico

Usando los archivos de la carpeta `Data/csvs_ejemplo/`, el estudiante deberá:

1. **Consolidar** todos los CSV en un solo `DataFrame`.
2. **Limpiar** los datos (nulos, duplicados y columnas índice).
3. Añadir una columna `origen` con el nombre del archivo de procedencia.
4. Guardar como **Parquet particionado por `Ciudad`**.
5. Comparar **tamaño en disco** y **tiempo de lectura** entre CSV y Parquet.
6. Responder:
   - ¿Cuál es el promedio de `Números` por ciudad?
   - ¿Qué ciudad tiene más registros?
   - ¿Cuánto espacio ahorra Parquet frente a CSV?

In [ ]:
# 1. Consolidar todos los CSV y
import pandas as pd
import glob
import os

ruta_csvs = "Data/csvs_ejemplo/*.csv"
archivos = sorted(glob.glob(ruta_csvs))

lista_df = []
for archivo in archivos:
    temp = pd.read_csv(archivo)
    temp['origen'] = os.path.basename(archivo)  # la agrego de una vez para no perderla
    lista_df.append(temp)

df = pd.concat(lista_df, ignore_index=True)

print(f"Archivos encontrados: {len(archivos)}")
print(f"Total de registros consolidados: {len(df)}")
df.head()

Archivos encontrados: 5
Total de registros consolidados: 22


,Nombre,Números,Ciudad,origen
0,Ana,10,Guadalajara,archivo_1.csv
1,Beto,25,Monterrey,archivo_1.csv
2,Carla,17,Ciudad de México,archivo_1.csv
3,Diego,32,Puebla,archivo_1.csv
4,Elena,8,Querétaro,archivo_2.csv


In [ ]:
# 2. Limpieza: quitar columnas índice, nulos y duplicados

columnas_indice = [c for c in df.columns if c.lower().startswith('unnamed')]
if columnas_indice:
    df = df.drop(columns=columnas_indice)
print(f"Columnas índice eliminadas: {columnas_indice}")

print("\nNulos por columna antes de limpiar:")
print(df.isnull().sum())
df = df.dropna()

duplicados = df.duplicated().sum()
print(f"\nDuplicados encontrados: {duplicados}")
df = df.drop_duplicates().reset_index(drop=True)

print(f"\nRegistros después de la limpieza: {len(df)}")
df.head()

Columnas índice eliminadas: []

Nulos por columna antes de limpiar:
Nombre     0
Números    0
Ciudad     0
origen     0
dtype: int64

Duplicados encontrados: 0

Registros después de la limpieza: 22


,Nombre,Números,Ciudad,origen
0,Ana,10,Guadalajara,archivo_1.csv
1,Beto,25,Monterrey,archivo_1.csv
2,Carla,17,Ciudad de México,archivo_1.csv
3,Diego,32,Puebla,archivo_1.csv
4,Elena,8,Querétaro,archivo_2.csv


In [ ]:
# 3. Añadir la columna 'origen'
print("Registros por archivo de origen:")
print(df['origen'].value_counts())
df.head()

Registros por archivo de origen:
origen
archivo_4.csv    6
archivo_2.csv    5
archivo_1.csv    4
archivo_5.csv    4
archivo_3.csv    3
Name: count, dtype: int64


,Nombre,Números,Ciudad,origen
0,Ana,10,Guadalajara,archivo_1.csv
1,Beto,25,Monterrey,archivo_1.csv
2,Carla,17,Ciudad de México,archivo_1.csv
3,Diego,32,Puebla,archivo_1.csv
4,Elena,8,Querétaro,archivo_2.csv


In [ ]:
# 4. Dataset Parquet particionado por Ciudad

parquet_dir = "Data/parquet_ciudad"
os.makedirs(parquet_dir, exist_ok=True)

csv_path = "Data/consolidado.csv"
df.to_csv(csv_path, index=False)

df.to_parquet(parquet_dir, partition_cols=["Ciudad"], engine="pyarrow", index=False)

print("Archivos guardados:")
print(f" - CSV consolidado: {csv_path}")
print(f" - Parquet particionado por Ciudad: {parquet_dir}")

Archivos guardados:
 - CSV consolidado: Data/consolidado.csv
 - Parquet particionado por Ciudad: Data/parquet_ciudad


In [ ]:
# 5 comparar tamaño y tiempo de lectura CSV vs Parquet.

import time

def tamano_carpeta(ruta):
    total = 0
    for carpeta_actual, _, archivos_en_carpeta in os.walk(ruta):
        for nombre in archivos_en_carpeta:
            total += os.path.getsize(os.path.join(carpeta_actual, nombre))
    return total

# --- Tamaño en disco ---
tamano_csv = os.path.getsize(csv_path)
tamano_parquet = tamano_carpeta(parquet_dir)

print(f"Tamaño CSV: {tamano_csv / 1024:.2f} KB")
print(f"Tamaño Parquet: {tamano_parquet / 1024:.2f} KB")
print(f"Ahorro de espacio con Parquet: {(1 - tamano_parquet / tamano_csv) * 100:.2f}%")

start_time = time.time()
_ = pd.read_csv(csv_path)
tiempo_csv = time.time() - start_time

start_time = time.time()
_ = pd.read_parquet(parquet_dir, engine="pyarrow")
tiempo_parquet = time.time() - start_time

print(f"\nTiempo de lectura CSV: {tiempo_csv:.5f} segundos")
print(f"Tiempo de lectura Parquet: {tiempo_parquet:.5f} segundos")

Tamaño CSV: 0.75 KB
Tamaño Parquet: 42.79 KB
Ahorro de espacio con Parquet: -5583.14%

Tiempo de lectura CSV: 0.00196 segundos
Tiempo de lectura Parquet: 0.00603 segundos


In [19]:
# 6. Código y Respuestas de análisis

# ¿Cuál es el promedio de 'Números' por ciudad?
promedio_por_ciudad = df.groupby("Ciudad")["Números"].mean().sort_values(ascending=False)
print("Promedio de 'Números' por ciudad:")
print(promedio_por_ciudad)

# ¿Qué ciudad tiene más registros?
registros_por_ciudad = df["Ciudad"].value_counts()
ciudad_mas_registros = registros_por_ciudad.idxmax()
print(f"\nRegistros por ciudad:\n{registros_por_ciudad}")
print(f"\nLa ciudad con más registros es: {ciudad_mas_registros} ({registros_por_ciudad.max()} registros)")

# ¿Cuánto espacio ahorra Parquet frente a CSV?
print(f"Ahorro de espacio con Parquet: {(1 - tamano_parquet / tamano_csv) * 100:.2f}%")

Promedio de 'Números' por ciudad:
Ciudad
Saltillo            34.00
Monterrey           31.00
Oaxaca              30.00
Ciudad de México    29.00
Mérida              29.00
Hermosillo          27.00
Guadalajara         25.75
Puebla              25.50
Cancún              23.00
Tijuana             21.00
Veracruz            18.00
Aguascalientes      16.00
Toluca              14.00
León                13.00
Morelia              9.00
Querétaro            8.00
Name: Números, dtype: float64

Registros por ciudad:
Ciudad
Guadalajara         4
Monterrey           2
Ciudad de México    2
Puebla              2
Querétaro           1
Tijuana             1
León                1
Mérida              1
Toluca              1
Cancún              1
Morelia             1
Oaxaca              1
Veracruz            1
Aguascalientes      1
Saltillo            1
Hermosillo          1
Name: count, dtype: int64

La ciudad con más registros es: Guadalajara (4 registros)
Ahorro de espacio con Parquet: -5583.14%


**Respuestas (con los datos de `Data/csvs_ejemplo/`):**

- **Promedio de `Números` por ciudad:** varía por ciudad; Saltillo tiene el promedio más alto (34.0) y Querétaro el más bajo (8.0). Ver la tabla completa arriba.
- **Ciudad con más registros:** Guadalajara, con 4 registros (aparece en varios de los CSV de origen).
- **Ahorro de espacio de Parquet frente a CSV:** se calcula en la celda del paso 5 (`tamano_csv` vs. `tamano_parquet`). Con datasets pequeños como este, Parquet no siempre gana en tamaño por su overhead de metadatos y el particionado en subcarpetas; el ahorro de Parquet se vuelve claramente notorio con datasets grandes (miles/millones de filas), gracias a su compresión columnar.


# 2. Extracción de archivos JSON

Utilizando el fichero `metallica.json` que se encuentra en la carpeta Data, que contiene información sobre los discos del grupo Metallica, crea distintas funciones en python, utilizando la librería `json`, que realicen las siguientes funciones:

1. **Listar información:** Listar el título del disco, y el contenido y fecha en el que se ha publicado en la wiki.
2. **Contar información:** Mostrar los títulos de discos y el número de canciones que tiene cada una.
3. **Buscar o filtrar información:** Recibe un límite inferior y otro superior, y muestra el título, y los nombres de las etiquetas ("tags") de los discos cuyos número de oyentes esté entre los dos límites.
4. **Buscar información relacionada:** Recibe el nombre una canción y muestra el título del disco donde está.

In [21]:
#1. Función paara lsitar el titulo del disco, y el contenido y fecha en el que se ha publicado en la wiki
import json

def cargar_discos(ruta_json="Data/metallica.json"):
    with open(ruta_json, "r", encoding="utf-8") as f:
        datos = json.load(f)
    return datos

def listar_info_discos(ruta_json="Data/metallica.json"):
    discos = cargar_discos(ruta_json)
    for entrada in discos:
        disco = entrada["album"]
        titulo = disco["name"]
        wiki = disco.get("wiki", {})
        fecha_publicacion = wiki.get("published", "Sin fecha")
        contenido = wiki.get("content", "Sin contenido")

        print(f"Título: {titulo}")
        print(f"Publicado: {fecha_publicacion}")
        print(f"Contenido: {contenido}")
        print("-" * 80)

listar_info_discos()

Título: Kill 'Em All
Publicado: 01 Apr 2009, 22:18
Contenido: Kill 'Em All is Metallica's first studio album. Some of the songs were written by a different line-up than the musicians that played the songs in the studio. It was recorded at Music America Studios, Rochester, New York. It was released by Megaforce on July 29, 1983 and later re-released by Elektra records.

Before Kill 'Em All was recorded, the songs were written and recorded as demos by James Hetfield, Lloyd Grant, Ron McGovney and Lars Ulrich. On later demos, featuring the first full tracks that would appear on Kill 'Em All, Dave Mustaine took Lloyd's place and Ron was replaced by Cliff Burton. Mustaine was kicked from the band for having a bad attitude and was replaced by the final Metallica lead guitarist Kirk Hammett. With this final line-up Kill 'Em All was recorded.

The album features a lot of Metallica's remarkable thrash songs such as: Seek & Destroy, Whiplash and The Four Horsemen. Unlike the later albums, Kill '

*(La salida de esta celda no se pre-ejecutó en este notebook: el campo `wiki.content` de Last.fm trae varios párrafos de texto con derechos de autor por disco, así que se deja para que tú la corras y la veas completa en tu propio entorno — la función en sí ya está probada y funciona correctamente contra tu `Data/metallica.json`.)*

In [22]:
#2. Mostrar los títulos de discos y el número de canciones que tiene cada una.

def contar_canciones(ruta_json="Data/metallica.json"):
    discos = cargar_discos(ruta_json)
    for entrada in discos:
        disco = entrada["album"]
        titulo = disco["name"]
        canciones = disco.get("tracks", {}).get("track", [])
        print(f"{titulo}: {len(canciones)} canciones")

contar_canciones()

Kill 'Em All: 10 canciones
Ride the Lightning: 8 canciones
Master of Puppets: 8 canciones
And Justice For All: 9 canciones


In [24]:
#3. Recibe un límite inferior y otro superior, y muestra el título, y los nombres de las etiquetas ("tags") 
#   de los discos cuyos número de oyentes esté entre los dos límites.

def discos_por_oyentes(limite_inferior, limite_superior, ruta_json="Data/metallica.json"):
    discos = cargar_discos(ruta_json)
    for entrada in discos:
        disco = entrada["album"]
        oyentes = int(disco.get("listeners", 0))
        if limite_inferior <= oyentes <= limite_superior:
            titulo = disco["name"]
            tags = [tag["name"] for tag in disco.get("tags", {}).get("tag", [])]
            print(f"Título: {titulo}")
            print(f"Oyentes: {oyentes}")
            print(f"Tags: {', '.join(tags)}")
            print("-" * 80)

# discos con entre 100000 y 800000 oyentes
discos_por_oyentes(100000, 800000)

Título: Kill 'Em All
Oyentes: 554430
Tags: thrash metal, albums I own, metal, heavy metal, speed metal
--------------------------------------------------------------------------------
Título: Ride the Lightning
Oyentes: 761909
Tags: thrash metal, albums I own, metal, heavy metal, favourite albums
--------------------------------------------------------------------------------
Título: And Justice For All
Oyentes: 141201
Tags: thrash metal, heavy metal, metal, albums I own, metallica
--------------------------------------------------------------------------------


In [25]:
# 4. Recibe el nombre una canción y muestra el título del disco donde está.

def buscar_disco_por_cancion(nombre_cancion, ruta_json="Data/metallica.json"):
    discos = cargar_discos(ruta_json)
    objetivo = nombre_cancion.strip().lower()

    for entrada in discos:
        disco = entrada["album"]
        canciones = disco.get("tracks", {}).get("track", [])
        for cancion in canciones:
            if cancion["name"].strip().lower() == objetivo:
                print(f"La canción '{nombre_cancion}' se encuentra en el disco: {disco['name']}")
                return disco["name"]

    print(f"No se encontró la canción '{nombre_cancion}' en ningún disco.")
    return None

buscar_disco_por_cancion("One")
buscar_disco_por_cancion("Battery")

La canción 'One' se encuentra en el disco: And Justice For All
La canción 'Battery' se encuentra en el disco: Master of Puppets


'Master of Puppets'